# Google Colab Version: [Open this notebook in Google Colab](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/quickstart.ipynb)

# Parawave Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/quickstart.ipynb)

This notebook walks through the core features of parawave. All examples use a mock API function — no API keys needed.

In [ ]:
!pip install parawave

In [ ]:
import parawave
import asyncio
import random


async def mock_api(prompt: str, fail_rate: float = 0.0, latency: float = 0.1) -> dict:
    """Simulates an LLM API call with configurable failure and latency."""
    await asyncio.sleep(random.uniform(0.01, latency))
    if random.random() < fail_rate:
        raise ConnectionError("API rate limit exceeded")
    words = ["the", "quick", "brown", "fox", "jumps", "over", "lazy", "dog"]
    return {
        "response": " ".join(random.choices(words, k=random.randint(5, 10))).capitalize() + ".",
        "tokens": random.randint(50, 200),
    }

## Part 1: Core Patterns (zero dependencies)

Everything in Part 1 works with just `pip install parawave` — no extras needed.

### 1a. Basic Run

Decorate an async function with `@parawave()` to turn it into a parallel job.
Call `.run(data=[...])` with a list of input dicts — each dict is unpacked as keyword arguments to the function.

In [ ]:
@parawave(max_concurrency=5, progress="console")
async def enrich(prompt: str) -> dict:
    return await mock_api(prompt)


cities = ["New York", "London", "Tokyo", "Paris", "Berlin",
          "Sydney", "Toronto", "Mumbai", "Seoul", "Cairo"]
data = [{"prompt": f"Tell me about {city}"} for city in cities]

result = enrich.run(data=data)

print("Summary:", result.summary)
print("All OK? ", result.ok)
print()
for i, city in enumerate(cities[:3]):
    print(f"  {city}: {result[i].output}")

### 1b. Retry with Backoff

Use `RetryPolicy` to automatically retry failed items with configurable backoff.
Here the mock API has a 30% failure rate — exponential backoff handles transient errors gracefully.

In [ ]:
from parawave import RetryPolicy


@parawave(
    max_concurrency=5,
    retry=RetryPolicy(max_retries=3, backoff="exponential", base_delay=0.1),
    progress="console",
)
async def enrich_flaky(prompt: str) -> dict:
    return await mock_api(prompt, fail_rate=0.3)


data = [{"prompt": f"Item {i}"} for i in range(15)]
result = enrich_flaky.run(data=data)

print("Summary:  ", result.summary)
print("Retried:  ", result.num_retried)
print("Attempts: ", result.num_attempts)

### 1c. Resume

When a run has failures, call `.resume()` to retry only the failed and pending items.
Loop until `result.ok` to drive everything to completion.

In [ ]:
@parawave(
    max_concurrency=5,
    retry=RetryPolicy(max_retries=1),
    progress="console",
)
async def enrich_unreliable(prompt: str) -> dict:
    return await mock_api(prompt, fail_rate=0.5)


data = [{"prompt": f"Item {i}"} for i in range(12)]
result = enrich_unreliable.run(data=data)
print(f"Round 1: {result.summary}")

attempt = 2
while not result.ok:
    result = enrich_unreliable.resume()
    print(f"Round {attempt}: {result.summary}")
    attempt += 1

print(f"\nDone in {attempt - 1} rounds. All OK? {result.ok}")

### 1d. Rate Limiting

`rate_limit=10` caps throughput at 10 items/sec — useful for respecting API rate limits.
Even with `max_concurrency=20`, the rate limiter throttles dispatch.

In [ ]:
import time


@parawave(max_concurrency=20, rate_limit=10, progress="console")
async def enrich_limited(prompt: str) -> dict:
    return await mock_api(prompt, latency=0.01)


data = [{"prompt": f"Item {i}"} for i in range(20)]

t0 = time.time()
result = enrich_limited.run(data=data)
elapsed = time.time() - t0

print(f"\nSummary: {result.summary}")
print(f"Wall time: {elapsed:.2f}s (rate_limit=10 means ~2s minimum for 20 items)")

### 1e. Hooks + SharedState

`SharedState` is a thread-safe dict for sharing data between parallel tasks and hooks.
Use `on_item_complete` to react to each completed item.

In [ ]:
from parawave import SharedState

state = SharedState({"total_tokens": 0, "count": 0})


def on_item_done(item):
    """Hook: accumulate token counts from each response."""
    state.increment("count")
    state.increment("total_tokens", item.output["tokens"])


@parawave(
    max_concurrency=5,
    on_item_complete=[on_item_done],
    progress="console",
)
async def enrich_tracked(prompt: str) -> dict:
    return await mock_api(prompt)


data = [{"prompt": f"Tell me about topic {i}"} for i in range(10)]
result = enrich_tracked.run(data=data)

print(f"\nSummary: {result.summary}")
print(f"Items processed: {state.get('count')}")
print(f"Total tokens:    {state.get('total_tokens')}")
print(f"State snapshot:  {state.to_dict()}")

## Part 2: Persistent Storage

Part 2 uses SQLite for persistent storage. Install with:

In [ ]:
!pip install parawave[sqlite]

### 2a. Same Function, Now Persistent

The only change from Part 1 is `storage="sqlite"`. Results are saved to disk — you can resume even after restarting Python.

In [ ]:
@parawave(max_concurrency=5, storage="sqlite", progress="console")
async def enrich_persistent(prompt: str) -> dict:
    return await mock_api(prompt)


cities = ["New York", "London", "Tokyo", "Paris", "Berlin"]
data = [{"prompt": f"Tell me about {city}"} for city in cities]

result = enrich_persistent.run(data=data)

print("Summary:", result.summary)
print("Run ID: ", result.run_id)
print()
for i, city in enumerate(cities):
    print(f"  {city}: {result[i].output['response'][:60]}...")

### 2b. RunManager

`RunManager` lets you browse and reload past runs. With SQLite, this works across sessions — restart your notebook and query past runs by ID.

In [ ]:
from parawave import RunManager


@parawave(max_concurrency=5, storage="sqlite", progress="console")
async def enrich_tagged(prompt: str) -> dict:
    return await mock_api(prompt)


# Run with tags so we can query later
data = [{"prompt": f"Item {i}"} for i in range(8)]
result = enrich_tagged.run(
    data=data,
    tags={"experiment": "quickstart", "stage": "generate"},
)
print("Run ID:", result.run_id)

# Create a RunManager pointing at the same SQLite storage
manager = RunManager("sqlite")

# List all stored runs
runs = manager.list_runs()
print(f"\nStored runs: {len(runs)}")
for run in runs:
    print(f"  {run.summary}")

# Get detailed info for our run
info = manager.get(result.run_id)
print(f"\nRun info:")
print(f"  Status:   {info.status}")
print(f"  Progress: {info.progress}")
print(f"  Tags:     {info.tags}")

# Load the full result back from storage
loaded = manager.load(result.run_id)
print(f"\nLoaded result: {loaded.summary}")
print(f"Outputs:       {loaded.data}")

### 2c. Cross-Session Resume

With `storage="sqlite"`, you can resume runs across sessions. If your process crashes, restart and call `.resume("run-id")` to pick up where you left off.

```python
# After restarting Python:
result = enrich_persistent.resume("run-abc123")  # picks up from where it left off
```

The run ID is printed after every run, and you can always find it via `RunManager("sqlite").list_runs()`.

## Key Points

- **`@parawave()`** turns any function into a parallel, resilient job
- **`RetryPolicy`** handles transient failures with configurable backoff
- **`.resume()`** retries only failed/pending items — loop until `result.ok`
- **`rate_limit`** throttles throughput to respect API rate limits
- **`SharedState`** + hooks enable real-time aggregation across parallel tasks
- **`storage="sqlite"`** persists results to disk for cross-session resume
- **`RunManager`** lets you browse, reload, and manage past runs

For more, see the [GitHub repo](https://github.com/parawaveio/parawave) and [documentation](https://parawave.io/docs).